In [6]:
import os
from pathlib import Path

# 0) Install deps (first time only)
%pip install -q kaggle python-dotenv

from dotenv import load_dotenv

# 1) Find and load .env (root of repo)
here = Path(".").resolve()
if (here / ".env").exists():
    env_path = here / ".env"
elif (here.parent / ".env").exists():
    env_path = here.parent / ".env"
else:
    raise FileNotFoundError("Could not find .env in . or ..")

load_dotenv(env_path)

# Sanity check
print("Loaded .env from:", env_path)
print("KAGGLE_USERNAME:", os.getenv("KAGGLE_USERNAME"))
print("KAGGLE_KEY set:", bool(os.getenv("KAGGLE_KEY")))

# 2) Download dataset into this notebook directory
dataset_ref = "valakhorasani/electric-vehicle-charging-patterns"
target_dir = "."  # current dir: ev_charging_sessions_2

!kaggle datasets download -d {dataset_ref} -p {target_dir} --unzip


Note: you may need to restart the kernel to use updated packages.
Loaded .env from: /Users/sho/projects/EvoCharge/.env
KAGGLE_USERNAME: qho162
KAGGLE_KEY set: True
Dataset URL: https://www.kaggle.com/datasets/valakhorasani/electric-vehicle-charging-patterns
License(s): apache-2.0
  0%|                                                | 0.00/130k [00:00<?, ?B/s]
100%|█████████████████████████████████████████| 130k/130k [00:00<00:00, 291MB/s]


In [ ]:
import pandas as pd

# change this to the actual filename if different
csv_path = "ev_charging_patterns.csv"

df = pd.read_csv(csv_path)

df.head()


((1320, 29), (1320,))

In [20]:
cols_to_drop_features = [
    "Charging Duration (hours)",
    "Charging End Time",
    "State of Charge (End %)",
    "Charging Cost (USD)",
    "User ID",
    "Charging Station ID",
    "Charging Start Time",
    "Distance Driven (since last charge) (km)",
    "Charging Rate (kW)",
]

df = df.drop(columns=cols_to_drop_features, errors="ignore")

# 2. Drop any remaining rows with NaNs (brute force, simple)
df = df.dropna()

# 3. One-hot encode categoricals
cols_to_ohe = [
    "Vehicle Model",
    "Charging Station Location",
    "Time of Day",
    "Day of Week",
    "Charger Type",
    "User Type",
]

df_enc = pd.get_dummies(df, columns=cols_to_ohe, drop_first=True)

In [16]:
# 1. How many NaNs per column?
na_counts = df.isna().sum().sort_values(ascending=False)
print(na_counts)

# 2. Which rows actually contain NaNs?
rows_with_nans = df[df.isna().any(axis=1)]
print("\nRows with NaNs:", rows_with_nans.shape[0])

# show a few problematic rows
rows_with_nans.head(10)

Vehicle Model                0
Battery Capacity (kWh)       0
Charging Station Location    0
Energy Consumed (kWh)        0
Time of Day                  0
Day of Week                  0
State of Charge (Start %)    0
Temperature (°C)             0
Vehicle Age (years)          0
Charger Type                 0
User Type                    0
dtype: int64

Rows with NaNs: 0


,Vehicle Model,Battery Capacity (kWh),Charging Station Location,Energy Consumed (kWh),Time of Day,Day of Week,State of Charge (Start %),Temperature (°C),Vehicle Age (years),Charger Type,User Type


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error


target_col = "Energy Consumed (kWh)"

# y = target
y = df_enc[target_col]

# X = everything except target
X = df_enc.drop(columns=[target_col])

print("X shape:", X.shape)
print("y shape:", y.shape)

# (optional but smart) drop any remaining rows with NaNs
mask = X.notna().all(axis=1) & y.notna()
X = X.loc[mask]
y = y.loc[mask]

# train / test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print("MAE:", mean_absolute_error(y_test, y_pred))
print("R^2:", r2_score(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))



X shape: (1254, 25)
y shape: (1254,)
MAE: 19.101228352101803
R^2: -0.08266813314237464
